# 05 — Peer Comparison
Builds a cosine-similarity peer mapping from normalized latest metrics.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
root=Path.cwd(); clean=root/'data'/'clean'; pl=pd.read_csv(clean/'profitandloss.csv'); bs=pd.read_csv(clean/'balancesheet.csv'); companies=pd.read_csv(clean/'companies.csv').merge(pd.read_csv(clean/'sector_mapping.csv'),on='symbol',how='left')
latest=lambda d:d.loc[~d.is_ttm.astype(bool)].sort_values('sort_order').groupby('symbol').tail(1)
frame=latest(pl)[['symbol','sales','net_profit','opm_pct','eps']].merge(latest(bs)[['symbol','debt_to_equity']],on='symbol').merge(companies[['symbol','sector']],on='symbol',how='left'); cols=['sales','net_profit','opm_pct','eps','debt_to_equity']; x=StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(frame[cols])); sim=cosine_similarity(x)
rows=[]
for i,symbol in enumerate(frame.symbol):
 candidates=[j for j in range(len(frame)) if j!=i and frame.sector.iloc[j]==frame.sector.iloc[i]]; candidates=sorted(candidates,key=lambda j:sim[i,j],reverse=True)[:5]
 rows.extend({'symbol':symbol,'peer_symbol':frame.symbol.iloc[j],'sector':frame.sector.iloc[i],'similarity':round(float(sim[i,j]),4),'rank':rank} for rank,j in enumerate(candidates,1))
peers=pd.DataFrame(rows); out=root/'data'/'warehouse'/'peer_mapping.csv'; peers.to_csv(out,index=False); display(peers.head(20)); print(out)